> [!Warning] 
> **This project is still in an early phase of development.**
>
> The [python API](../api.html) is not yet stable, and some aspects of the schema for the [blueprint](../terminology.html#term-blueprint) will likely evolve. 
> Therefore whilst you are welcome to try out using the package, we cannot yet guarantee backwards compatibility. 
We expect to reach a more stable version in Q1 2025.
>
> To see which systems C-Star has been tested on so far, see [Supported Systems](../machines.html).

# Working with the `AdditionalCode` class

## Contents
1. [Introduction](#1.-Introduction)
2. [Working with local code](#2.-Working-with-local-code)
3. [Working with remote code](#3.-Working-with-remote-code)

## 1. Introduction
[(return to top)](#Contents)

`AdditionalCode` instances hold collections of related code, either in a local directory or remote repository. In this guide we will explore the structure and methods of the `AdditionalCode` class.

## 2. Working with local code
[(return to top)](#Contents)

If the code you intend to work with already exists on your local system, instantiating an `AdditionalCode` instance is straightforward: provide a path to a directory containing the code, and a list of files:

In [1]:
import shutil

from pathlib import Path
from cstar.base import AdditionalCode

# Create a directory to simulate pre-existing code.
output_path = Path("../../examples/outputs/additional_code_example")
shutil.rmtree(output_path, ignore_errors=True)

# Populate the simulated pre-existing code directory.
local_code = output_path / "local_code"
local_code.mkdir(parents=True)

In [2]:
# Populate the simulated pre-existing code directory.
!git clone https://github.com/CESR-lab/ucla-roms {local_code}/my_ucla_roms

Cloning into '../../examples/outputs/additional_code_example/local_code/my_ucla_roms'...
remote: Enumerating objects: 9556, done.
remote: Counting objects: 100% (659/659), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 9556 (delta 531), reused 510 (delta 499), pack-reused 8897 (from 2)
Receiving objects: 100% (9556/9556), 22.40 MiB | 12.82 MiB/s, done.
Resolving deltas: 100% (7151/7151), done.


In [ ]:
# Instantiate AdditionalCode using the local directory
compile_time_code = AdditionalCode(
    location=local_code / "my_ucla_roms/Examples/Rivers_Real",
    files=[
        "cppdefs.opt",
        "flux_frc.opt",
        "ocean_vars.opt",
        "param.opt",
        "river_frc.opt",
    ],
)
print(compile_time_code)

AdditionalCode
--------------
Location: ../../examples/outputs/additional_code_example/local_code/my_ucla_roms/Examples/Rivers_Real
Subdirectory: 
Working path: None
Exists locally: False (get with AdditionalCode.get())
Files:
    cppdefs.opt
    flux_frc.opt
    ocean_vars.opt
    param.opt
    river_frc.opt


<div class="alert alert-info">

**Note**

We see that "Exists locally" is `False`, in this context referring to the existence of a local copy that C-Star can work with. C-Star will not attempt to tamper with the original source of any files, and instead works with local copies. A local copy that C-Star can work with can be established using `AdditionalCode.get()`
</div>

### Fetching a working copy of the code with `AdditionalCode.get()`:

In [4]:
compile_time_code.get(local_dir=output_path)

2025-04-16 16:10:53,501 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying cppdefs.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example
2025-04-16 16:10:53,502 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying flux_frc.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example
2025-04-16 16:10:53,503 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying ocean_vars.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example
2025-04-16 16:10:53,503 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying param.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example
2025-04-16 16:10:53,504 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying river_frc.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example


In [5]:
print(compile_time_code)

AdditionalCode
--------------
Location: ../../examples/outputs/additional_code_example/local_code/my_ucla_roms/Examples/Rivers_Real
Subdirectory: 
Working path: /Users/chris/code/cstar/examples/outputs/additional_code_example
Exists locally: True
Files:
    cppdefs.opt
    flux_frc.opt
    ocean_vars.opt
    param.opt
    river_frc.opt


... We now see that we have a source location, as before, and a working copy, at the `Working path`. `Exists locally` is now also `True`.

## 3. Working with remote code
[(return to top)](#Contents)

C-Star can also work with code that is stored in a remote repository, using git. The process of creating an `AdditionalCode` instance in this case involves a couple of additional parameters:
- `checkout_target`: the specific point in the history of the repository to work worth. This can be either a branch, tag, or commit hash. 
- `subdir`: the subdirectory of the repository containing the files

Let's create the same `AdditionalCode` as above, but with a remote source this time:

In [6]:
compile_time_code = AdditionalCode(
    location="https://github.com/CESR-lab/ucla-roms.git",
    files=[
        "cppdefs.opt",
        "flux_frc.opt",
        "ocean_vars.opt",
        "param.opt",
        "river_frc.opt",
    ],
    subdir="Examples/Rivers_real",
    checkout_target="main",
)
print(compile_time_code)

AdditionalCode
--------------
Location: https://github.com/CESR-lab/ucla-roms.git
Subdirectory: Examples/Rivers_real
Checkout target: main
Working path: None
Exists locally: False (get with AdditionalCode.get())
Files:
    cppdefs.opt
    flux_frc.opt
    ocean_vars.opt
    param.opt
    river_frc.opt


### Fetching a working copy of the code with `AdditionalCode.get()`. 

For a remote repository, C-Star clones the repository to a temporary directory, then copies the desired files to the local target directory:

In [7]:
compile_time_code.get(local_dir=output_path / "remote_ucla_roms")

Cloning `https://github.com/CESR-lab/ucla-roms.git`


2025-04-16 16:10:56,088 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying cppdefs.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example/remote_ucla_roms
2025-04-16 16:10:56,089 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying flux_frc.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example/remote_ucla_roms
2025-04-16 16:10:56,090 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying ocean_vars.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example/remote_ucla_roms
2025-04-16 16:10:56,091 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying param.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example/remote_ucla_roms
2025-04-16 16:10:56,091 | INFO | cstar.base.additional_code.AdditionalCode::get:209: Copying river_frc.opt to /Users/chris/code/cstar/examples/outputs/additional_code_example/remote_ucla_roms


Cloned https://github.com/CESR-lab/ucla-roms.git to /var/folders/pj/bk3042bs0tldhwq8htgfnm640000gn/T/tmpmdok71s1
Checking out `https://github.com/CESR-lab/ucla-roms.git` @ `main`
Checked out main in git repository /var/folders/pj/bk3042bs0tldhwq8htgfnm640000gn/T/tmpmdok71s1


In [8]:
print(compile_time_code)

AdditionalCode
--------------
Location: https://github.com/CESR-lab/ucla-roms.git
Subdirectory: Examples/Rivers_real
Checkout target: main
Working path: /Users/chris/code/cstar/examples/outputs/additional_code_example/remote_ucla_roms
Exists locally: True
Files:
    cppdefs.opt
    flux_frc.opt
    ocean_vars.opt
    param.opt
    river_frc.opt


In [9]:
if 0:
    # clean up outputs when done
    import shutil

    shutil.rmtree(output_path, ignore_errors=True)